# Câu 1 - Thuật toán Greedy Best First Search

Tìm đường thoát từ phòng trung tâm ra rìa lâu đài trên ma trận `n x n`. Hai ô chỉ nối với nhau nếu có chung cạnh. Ô có giá trị `1` là đi được, ô có giá trị `0` là không đi được.

File đầu vào dùng trong bài là `A_in.csv`. Notebook này trình bày riêng thuật toán Greedy Best First Search.

## Đề bài và dữ liệu đầu vào

Dòng đầu của `A_in.csv` là:

```text
10,5,5
```

Ý nghĩa:

- `n = 10`: lâu đài là ma trận `10 x 10`.
- Vị trí bắt đầu là `(5,5)`.
- Tọa độ dùng kiểu `0-based`, tức dòng/cột bắt đầu từ `0`.

Mục tiêu là tìm một đường đi từ `(5,5)` đến một ô bất kỳ nằm trên rìa ma trận.

## Định nghĩa khoảng cách Manhattan và công thức h(x)

Khoảng cách Manhattan là khoảng cách di chuyển trên lưới khi chỉ được đi theo các hướng ngang hoặc dọc, không được đi chéo.

Với hai ô:

```text
A = (x1, y1)
B = (x2, y2)
```

khoảng cách Manhattan giữa `A` và `B` là:

```text
d(A, B) = |x1 - x2| + |y1 - y2|
```

Trong bài lâu đài, mục tiêu không phải một ô cố định mà là bất kỳ ô nào nằm trên rìa ma trận. Với một ô hiện tại:

```text
x = (row, col)
```

rìa lâu đài gồm bốn cạnh:

- rìa trên: `row = 0`
- rìa trái: `col = 0`
- rìa dưới: `row = n - 1`
- rìa phải: `col = n - 1`

Khoảng cách Manhattan ngắn nhất từ ô `x` đến rìa gần nhất là:

```text
h(x) = min(
    |row - 0|,
    |col - 0|,
    |row - (n - 1)|,
    |col - (n - 1)|
)
```

Vì `row`, `col` đều nằm trong ma trận nên có thể viết gọn:

```text
h(x) = min(row, col, n - 1 - row, n - 1 - col)
```

Ví dụ với `n = 10`, ô bắt đầu `(5,5)`:

```text
Đến rìa trên:  |5 - 0| = 5
Đến rìa trái:  |5 - 0| = 5
Đến rìa dưới:  |5 - 9| = 4
Đến rìa phải:  |5 - 9| = 4

h(5,5) = min(5, 5, 4, 4) = 4
```

Nếu xét ô `(7,9)`:

```text
Đến rìa phải: |9 - 9| = 0
h(7,9) = 0
```

Vì `h(7,9) = 0`, ô `(7,9)` nằm trên rìa và là một cửa ra hợp lệ.

Trong code, công thức này được cài đặt bằng hàm:

```python
def heuristic_to_border(position, n):
    row, col = position
    distances = [
        abs(row - 0),
        abs(col - 0),
        abs(row - (n - 1)),
        abs(col - (n - 1)),
    ]
    return min(distances)
```

## Biểu diễn bài toán dưới dạng đồ thị

Bài toán lâu đài có thể xem như một bài toán tìm đường trên đồ thị:

- Mỗi ô trong ma trận là một đỉnh.
- Chỉ những ô có giá trị `1` mới là đỉnh hợp lệ vì đó là ô có đường hầm.
- Hai đỉnh có cạnh nối với nhau nếu hai ô tương ứng kề cạnh theo một trong bốn hướng: trên, phải, dưới, trái.
- Ô bắt đầu là `(5,5)`.
- Tập đích là các ô đi được nằm trên rìa ma trận.

Chương trình không tạo sẵn toàn bộ danh sách cạnh. Thay vào đó, khi đang xét một ô, hàm `get_neighbors()` sinh ra các ô kề hợp lệ. Vì vậy đồ thị được suy ra trực tiếp từ ma trận đầu vào.

Ví dụ, từ ô `(5,5)`, chương trình xét các ô `(4,5)`, `(5,6)`, `(6,5)`, `(5,4)`, rồi chỉ giữ lại các ô nằm trong ma trận và có giá trị `1`.

## Code chương trình Greedy Best First Search

Dưới đây là chương trình hiện có trong file `cau1_greedy_best_first.py`. Notebook chỉ sao chép nội dung để trình bày, không chỉnh sửa file code gốc.

In [ ]:
from pathlib import Path
import csv
import heapq

import matplotlib.pyplot as plt


def read_input(file_path):
    with open(file_path, newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        first_line = next(reader)
        n = int(first_line[0])
        start = (int(first_line[1]), int(first_line[2]))
        maze = [[int(value) for value in row] for row in reader]

    return n, start, maze


def heuristic_to_border(position, n):
    row, col = position
    distances = [
        abs(row - 0),
        abs(col - 0),
        abs(row - (n - 1)),
        abs(col - (n - 1)),
    ]
    return min(distances)


def is_inside(position, n):
    row, col = position
    return 0 <= row < n and 0 <= col < n


def is_border(position, n):
    row, col = position
    return row == 0 or row == n - 1 or col == 0 or col == n - 1


def get_neighbors(position, n, maze):
    row, col = position
    directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    neighbors = []

    for d_row, d_col in directions:
        next_pos = (row + d_row, col + d_col)

        if is_inside(next_pos, n) and maze[next_pos[0]][next_pos[1]] == 1:
            neighbors.append(next_pos)

    return neighbors


def reconstruct_path(parent, goal):
    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()
    return path


def greedy_escape(n, start, maze):
    if not is_inside(start, n) or maze[start[0]][start[1]] != 1:
        return None

    frontier = []
    start_h = heuristic_to_border(start, n)
    heapq.heappush(frontier, (start_h, 0, start))

    parent = {start: None}
    visited = set()
    order = 0

    while frontier:
        _, _, current = heapq.heappop(frontier)

        if current in visited:
            continue

        visited.add(current)

        if is_border(current, n):
            return reconstruct_path(parent, current)

        for neighbor in get_neighbors(current, n, maze):
            if neighbor not in visited and neighbor not in parent:
                parent[neighbor] = current
                order += 1
                h = heuristic_to_border(neighbor, n)
                heapq.heappush(frontier, (h, order, neighbor))

    return None


def write_output(file_path, path):
    with open(file_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)

        if path is None:
            writer.writerow([-1])
            return

        writer.writerow([len(path)])
        writer.writerows(path)


def save_path_chart(maze, path, output_file):
    n = len(maze)

    plt.figure(figsize=(6, 6))
    plt.imshow(maze, cmap="gray_r")
    plt.xticks(range(n))
    plt.yticks(range(n))
    plt.grid(color="lightgray", linewidth=0.5)

    if path is not None:
        rows = [position[0] for position in path]
        cols = [position[1] for position in path]
        plt.plot(cols, rows, color="red", linewidth=2.5, marker="o", markersize=5, label="Duong di Greedy Manhattan")
        plt.scatter(cols[0], rows[0], c="lime", s=140, edgecolors="black", label="Bat dau")
        plt.scatter(cols[-1], rows[-1], c="yellow", s=140, edgecolors="black", label="Cua ra")

    plt.title("Duong thoat tim duoc bang Greedy voi h Manhattan")
    plt.xlabel("Cot")
    plt.ylabel("Dong")
    plt.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.savefig(output_file, dpi=160)
    plt.close()


def main():
    current_dir = Path(__file__).resolve().parent
    input_file = current_dir.parent / "A_in.csv"
    output_file = current_dir / "Greedy_out.csv"
    path_image = current_dir / "cau1_greedy_path.png"

    n, start, maze = read_input(input_file)
    path = greedy_escape(n, start, maze)
    write_output(output_file, path)
    save_path_chart(maze, path, path_image)

    if path is None:
        print("Greedy khong tim thay duong thoat.")
    else:
        print(f"Greedy tim thay duong thoat: {len(path)} o.")
        print(f"Da ghi ket qua vao: {output_file}")
        print(f"Da luu bieu do duong di: {path_image}")


if __name__ == "__main__":
    main()

## Giải thích tác dụng của từng hàm

- `read_input(file_path)`: đọc file CSV đầu vào, lấy kích thước ma trận `n`, vị trí bắt đầu `start` và ma trận lâu đài `maze`.
- `heuristic_to_border(position, n)`: tính khoảng cách Manhattan ngắn nhất từ ô hiện tại đến rìa gần nhất. Greedy dùng giá trị này để ưu tiên ô có vẻ gần cửa ra nhất.
- `is_inside(position, n)`: kiểm tra một tọa độ có nằm trong phạm vi ma trận `n x n` hay không.
- `is_border(position, n)`: kiểm tra một ô có nằm ở rìa lâu đài hay không. Nếu có, ô đó là cửa ra hợp lệ.
- `get_neighbors(position, n, maze)`: sinh các ô kề cạnh hợp lệ theo bốn hướng trên, phải, dưới, trái. Chỉ những ô nằm trong ma trận và có giá trị `1` mới được đi tiếp.
- `reconstruct_path(parent, goal)`: truy vết đường đi từ ô cửa ra về ô bắt đầu dựa trên dictionary `parent`, sau đó đảo ngược lại để có đường đi đúng chiều.
- `greedy_escape(n, start, maze)`: thực hiện Greedy Best First Search bằng hàng đợi ưu tiên theo heuristic `h`. Hàm luôn chọn ô có vẻ gần rìa nhất trước.
- `write_output(file_path, path)`: ghi kết quả ra file CSV. Nếu không tìm thấy đường thì ghi `-1`, nếu tìm thấy thì ghi số ô của đường đi và danh sách tọa độ.
- `save_path_chart(maze, path, output_file)`: vẽ ma trận lâu đài và đường đi tìm được, sau đó lưu thành ảnh PNG.
- `main()`: hàm điều phối chính, đọc input, chạy thuật toán, ghi output, lưu biểu đồ và in thông báo kết quả.

## Giải thích thuật toán Greedy Best First Search

Greedy Best First Search là thuật toán tìm kiếm có thông tin. Thuật toán dùng heuristic để đo xem trạng thái nào có vẻ gần mục tiêu nhất, rồi ưu tiên xét trạng thái đó trước.

Trong bài này, heuristic là khoảng cách Manhattan ngắn nhất từ ô hiện tại đến rìa gần nhất:

```text
h(x) = min(row, col, n - 1 - row, n - 1 - col)
```

Greedy chỉ dùng `h(x)` để chọn ô tiếp theo. Khác với A*, Greedy không cộng thêm chi phí đã đi `g(x)`.

Quy trình thuật toán trong code:

1. Kiểm tra ô bắt đầu `(5,5)` có hợp lệ và đi được không. Nếu không hợp lệ thì trả về `None`.
2. Khởi tạo `frontier` là hàng đợi ưu tiên. Đưa ô bắt đầu vào với độ ưu tiên là `h(start)`.
3. Khởi tạo `parent = {start: None}` để lưu đường đi và `visited` để lưu các ô đã xử lý chính thức.
4. Trong vòng lặp, lấy ô có `h` nhỏ nhất ra khỏi `frontier` bằng `heapq.heappop()`.
5. Nếu ô đó đã có trong `visited`, bỏ qua để tránh xử lý lại.
6. Nếu ô vừa lấy ra nằm trên rìa ma trận, gọi `reconstruct_path()` để dựng đường đi và kết thúc.
7. Nếu chưa tới rìa, gọi `get_neighbors()` để sinh các ô kề hợp lệ.
8. Với mỗi ô kề chưa xử lý và chưa được phát hiện trong `parent`, lưu cha của nó, tính `h`, rồi đưa vào `frontier`.
9. Nếu `frontier` rỗng mà chưa gặp ô rìa, nghĩa là không tìm thấy đường thoát.

Mô tả vòng lặp Greedy:

```text
Lặp khi frontier chưa rỗng:
    lấy ô có h nhỏ nhất
    nếu ô đó là rìa -> tìm thấy cửa ra
    nếu chưa -> thêm các ô kề hợp lệ vào frontier theo h
```

Greedy thường đi nhanh về phía rìa vì luôn ưu tiên ô có `h` nhỏ. Tuy nhiên do không xét chi phí đã đi, Greedy có thể chọn một hướng nhìn gần mục tiêu nhưng không phải đường ngắn nhất. Trong dữ liệu này, Greedy tìm được đường đi gồm `7` ô, trùng với đường đi A*.

## Phân tích chi tiết luồng xử lý

Trước tiên, chương trình kiểm tra ô bắt đầu `(5,5)` có nằm trong ma trận và có đi được không. Nếu ô bắt đầu không hợp lệ thì trả về `None`.

Sau đó thuật toán khởi tạo cấu trúc dữ liệu chính: `frontier` là hàng đợi ưu tiên theo `h`. `visited` lưu các ô đã xử lý chính thức. `parent` lưu cha của từng ô để truy vết đường đi.

Ở mỗi vòng lặp, thuật toán lấy một ô ra xử lý, kiểm tra ô đó có nằm trên rìa không. Nếu đã ở rìa, chương trình gọi `reconstruct_path()` để dựng lại đường đi. Nếu chưa tới rìa, chương trình gọi `get_neighbors()` để sinh các ô kề hợp lệ rồi đưa các ô phù hợp vào cấu trúc chờ xét.

## Bảng bước chạy chi tiết

| Bước | Ô lấy ra | h | Thêm vào frontier | Frontier sau bước | Ghi chú |
| --- | --- | --- | --- | --- | --- |
| 1 | `(5,5)` | 4 | `(4,5) h=4`, `(5,6) h=3` | `[(5,6) h=3, (4,5) h=4]` | Tiếp tục |
| 2 | `(5,6)` | 3 | `(5,7) h=2` | `[(5,7) h=2, (4,5) h=4]` | Tiếp tục |
| 3 | `(5,7)` | 2 | `(5,8) h=1` | `[(5,8) h=1, (4,5) h=4]` | Tiếp tục |
| 4 | `(5,8)` | 1 | `(4,8) h=1`, `(6,8) h=1` | `[(4,8) h=1, (6,8) h=1, (4,5) h=4]` | Tiếp tục |
| 5 | `(4,8)` | 1 | `(3,8) h=1` | `[(6,8) h=1, (3,8) h=1, (4,5) h=4]` | Tiếp tục |
| 6 | `(6,8)` | 1 | `(7,8) h=1` | `[(3,8) h=1, (7,8) h=1, (4,5) h=4]` | Tiếp tục |
| 7 | `(3,8)` | 1 | `(2,8) h=1` | `[(7,8) h=1, (2,8) h=1, (4,5) h=4]` | Tiếp tục |
| 8 | `(7,8)` | 1 | `(7,9) h=0` | `[(7,9) h=0, (2,8) h=1, (4,5) h=4]` | Tiếp tục |
| 9 | `(7,9)` | 0 |  | `[(2,8) h=1, (4,5) h=4]` | Gặp cửa ra |

## Biểu đồ đường đi

![Đường thoát tìm được](cau1_greedy_path.png)

Trong hình minh họa:

- Ô có giá trị `1` là ô đi được.
- Ô có giá trị `0` là tường hoặc không có đường hầm.
- Đường màu đỏ nối các tọa độ trong danh sách đường đi.
- Điểm đầu là `(5,5)`, tức phòng trung tâm.
- Điểm cuối là ô nằm trên rìa ma trận, tức cửa ra.

Đường đi trong file output là:

`(5,5)` -> `(5,6)` -> `(5,7)` -> `(5,8)` -> `(6,8)` -> `(7,8)` -> `(7,9)`

Dòng đầu của file output là `7`, nghĩa là đường đi có `7` ô. Nếu tính số bước di chuyển thì số bước bằng `7 - 1`.

## Kết quả thực thi

Lệnh chạy chương trình:

```powershell
python 2025/De1/BFS_DFS_Astar_Manhattan_Cau1/04_Greedy_BestFirst/cau1_greedy_best_first.py
```

Kết quả trong `Greedy_out.csv`:

```text
7
5,5
5,6
5,7
5,8
6,8
7,8
7,9
```

Kết luận: Greedy Best First Search tìm được đường thoát hợp lệ từ `(5,5)` đến một ô ở rìa ma trận.